# Run inference a pre-trained model

## Load a pre-trained model from checkpoints (previous notebook)

In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx

import orbax
from orbax import checkpoint
from jax.sharding import SingleDeviceSharding

from src.model import NanoLLM
from src.inference import generate_text
from src.config import ModelConfig, model_config, TokenizerConfig, CHECKPOINTS_DIR

In [ ]:
# import model
model = NanoLLM()

In [ ]:
cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)

checkpoint_path = CHECKPOINTS_DIR / "nano_checkpoint.orbax"
checkpointer = orbax.checkpoint.PyTreeCheckpointer()

In [ ]:
# TODO: extract to these to src directory
restore_args = jax.tree_util.tree_map(
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),
    nnx.state(model)
)

restored_state = checkpointer.restore(
    checkpoint_path,
    item=nnx.state(model),
    restore_args=restore_args)

In [ ]:
# Update the model with the loaded checkpoints
nnx.update(model,restored_state)

## Run inference on the pre-trained model

In [ ]:
# Use tokenizer from config
tokenizer_config = TokenizerConfig(delimiter="<|endoftext|>", name="gpt2")
tokenizer = tokenizer_config.tokenizer
DELIMITER = tokenizer_config.delimiter  # Use the same delimiter from config

# Define method for creating a story
def create_story(story_prompt, max_new_tokens, temperature):

    # Convert the text prompt to token IDs using the tokenizer
    start_tokens = tokenizer.encode(story_prompt)
    
    # TODO: Save tokenizer and delimiter variables in a config file alongside the checkpoints because they are 
    # aspects of how the model was trained. We can't pass in any tokenizer or any delimiter here.
    # NB: But we cannot couple the tokenizer and delimiter to the model itself b/c they might change if we retrain the model.
    return generate_text(model, tokenizer, DELIMITER, start_tokens, max_new_tokens, temperature)

In [ ]:
# Call the method to exercise inference
# The result at this time is super unsatisfactory because the model was not trained very long 
# and the inference method is not optimized.
create_story("I was eating a bowl of rasberries when I noticed that", 30, 0.2)